# Streaming Responses + Token Counting

## Problem Statement
Build a streaming SSIS error log analyzer with per-session token cost tracking.

## My Approach
First with help of functions enable streaming live, and get response written, meanwhile capture usage for it and then print it at the end.

## What I Learned
- Streaming returns a generator, not a response object — you iterate chunks, not wait for one big reply.
- The `usage` data only lives on the terminal chunk (when `finish_reason` is not None). Every earlier chunk has it as null — so you must stay in the loop until the end.
- `full_response` must be assembled manually by concatenating deltas — the stream does not give you a final assembled string.
- The session ledger belongs outside the per-log function. A function that prints its own summary cannot accumulate state across multiple calls.
- Groq exposes usage via `chunk.x_groq.usage` or `chunk.usage` — worth checking both with getattr as a safety net.

## Where I Got Stuck
- Printed `full_response` after the loop thinking it was the ledger — it just double-printed the streamed output. The fix was to return `(full_response, usage_data)` and let the caller own the printing.
- A separate `run_session` function that loops all logs and prints the summary once at the end.

## What I'd Do Differently
- Design the function boundary first: one function streams and returns data, one function owns the session loop and printing. Mixing both into one function caused the double-print bug.
- Wire up a real cost constant from day one, even if it's $0.00 - makes swapping to a paid model a one-line change instead of a refactor.

In [ ]:
from groq import Groq
import os
from dotenv import load_dotenv


load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "llama-3.1-8b-instant"  

In [12]:
SAMPLE_LOGS = [
    f'''Error: SSIS Package 'Load_Customer_Dim' failed at Data Flow Task. OLE DB Destination [147]: Cannot insert NULL into column 'customer_id'. Rows rejected: 412. Execution time: 00:03:22.''',

    f'''Warning: Flat File Source [23] detected encoding mismatch on file .'sales_export_20240301.csv'. Expected UTF-8, found Windows-1252.Rows affected: 1,847.''',

    f'''Error: Execute SQL Task 'Truncate_Staging' failed.Connection to SQL Server instance 'PROD-ETL-01' timed out after 30s.Retry attempts: 3. All failed.'''
]

In [40]:
def few_shot_prompt(log_text):
    return f"""Review SSIS error and answer in below format .
Example:-
Input : "Error: SSIS Package 'Load_Customer_Dim' failed at Data Flow Task. "
"OLE DB Destination [147]: Cannot insert NULL into column 'customer_id'. "
"Rows rejected: 412. Execution time: 00:03:22.",

Output : 
**Error Type:** OLE DB Destination failure — NULL constraint violation
**Root Cause:** 412 source rows are arriving with a NULL `customer_id`. This is a data quality issue upstream of the SSIS package, likely in the source query or a preceding transformation that failed to filter/default the key column.
**Immediate Action:** Check the source OLE DB query — add a `WHERE customer_id IS NOT NULL` filter or trace which upstream table is producing NULLs. Do not re-run the package until the source is fixed or rows will be rejected again.
**Severity:** High — 412 rows silently dropped from Customer dimension load.

Stick to below schema only , no additional note or column nothing .
Return answer with below schema :- 
    Error Type: str
    Root Cause: str
    Immediate Action: str
    Severity: Literal["critical", "high", "medium", "low"]

Log:-{log_text}
"""

## My Solution (Naive)
*First attempt — written before reviewing feedback*

In [41]:
COST_PER_1K_TOKENS = 0.00

def parse_ssis_log(log_text: str, promptbuilder):
    prompt = promptbuilder(log_text)

    response_stream = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a senior DBA specializing in SSIS pipeline triage."},
            {"role": "user", "content": prompt}
        ],
        stream=True
    )

    full_response = ""
    usage_data = None

    for chunk in response_stream:
        if not chunk.choices:
            continue

        choice = chunk.choices[0]
        delta = choice.delta.content or ""
        print(delta, end="", flush=True)
        full_response += delta

        if choice.finish_reason is not None:
            usage_data = (
                getattr(getattr(chunk, "x_groq", None), "usage", None) or getattr(chunk, "usage", None)
            )

    print()
    print("\n=== Session Token Ledger ===")
    print(full_response)

    print("\n--------------------------------")
    print(usage_data)


## Refactored Solution

In [42]:
COST_PER_1K_TOKENS = 0.00

def parse_ssis_log(log_text: str, promptbuilder):
    prompt = promptbuilder(log_text)

    response_stream = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a senior DBA specializing in SSIS pipeline triage."},
            {"role": "user", "content": prompt}
        ],
        stream=True
    )

    full_response = ""
    usage_data = None

    for chunk in response_stream:
        if not chunk.choices:
            continue

        choice = chunk.choices[0]
        delta = choice.delta.content or ""
        print(delta, end="", flush=True)
        full_response += delta

        if choice.finish_reason is not None:
            usage_data = (
                getattr(getattr(chunk, "x_groq", None), "usage", None) or getattr(chunk, "usage", None)
            )

    print()
    return full_response, usage_data


In [ ]:

def run_session(logs, prompt):
    ledger = []

    for i, log in enumerate(logs):
        print(f"\n=== Analyzing Log {i+1}/{len(logs)} ===")
        _, usage = parse_ssis_log(log, prompt)

        ledger.append({
            "prompt": usage.prompt_tokens,
            "completion": usage.completion_tokens,
            "total": usage.total_tokens
        })

    # Print ledger after all logs
    print("\n=== Session Token Ledger ===")
    for i, entry in enumerate(ledger):
        print(f"Log {i+1}: {entry['prompt']} prompt | {entry['completion']} completion | {entry['total']} total")

    print("─" * 45)
    total_prompt = sum(e["prompt"] for e in ledger)
    total_completion = sum(e["completion"] for e in ledger)
    total_tokens = sum(e["total"] for e in ledger)
    est_cost = (total_tokens / 1000) * COST_PER_1K_TOKENS

    print(f"TOTAL:  {total_prompt} prompt | {total_completion} completion | {total_tokens:,} total")
    print(f"Est. cost: ${est_cost:.4f} (Groq free tier)")


In [ ]:
run_session(SAMPLE_LOGS,few_shot_prompt)


=== Analyzing Log 1/3 ===
**Error Type:** OLE DB Destination failure — NULL constraint violation
**Root Cause:** 412 source rows are arriving with a NULL `customer_id`. This is a data quality issue upstream of the SSIS package, likely in the source query or a preceding transformation that failed to filter/default the key column.
**Immediate Action:** Check the source OLE DB query — add a `WHERE customer_id IS NOT NULL` filter or trace which upstream table is producing NULLs. Do not re-run the package until the source is fixed or rows will be rejected again.
**Severity:** High — 412 rows silently dropped from Customer dimension load.

=== Analyzing Log 2/3 ===
**Error Type:** Flat File Source failure — encoding mismatch
**Root Cause:** The file `sales_export_20240301.csv` has a detected encoding mismatch, expecting UTF-8 but found Windows-1252.
**Immediate Action:** Review the source file to ensure correct encoding and manually adjust the Flat File Source properties to match the actual